# 🔌 MCP Agents — Build a Server Anyone Can Connect To

> **HackAI 2026 — Master's track** · Notebook 2/8 · ⏱️ 75 minutes · 🏆 100 pts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

The **Model Context Protocol (MCP)** is to LLM tools what HTTP is to the web: a single open standard so *any* agent (Claude, Cursor, Continue, Cline, smolagents, LangChain…) can call *any* tool you publish.

By end of this notebook you will:
1. Understand the MCP architecture (host ↔ client ↔ server)
2. Ship a **production-quality MCP server** in Python with `FastMCP`
3. Expose **resources, tools, and prompts** — the three MCP primitives
4. Connect it to **smolagents** AND to Claude Desktop
5. Build a **Darija toolkit MCP server** for the leaderboard challenge

### 🏆 Challenge — *"Plug & Play"*
Publish an MCP server that wraps **at least 3 useful Darija/Arabic tools**. The mentor's evaluation agent will connect to your server URL and run a hidden test set. Top accuracy + cleanest schema = 🥇.

---


## 0 · The MCP mental model in one picture

```
┌─────────────┐    JSON-RPC over stdio/SSE/HTTP    ┌──────────────┐
│   HOST      │ ◀──────────────────────────────▶  │    SERVER    │
│ (Claude     │      list_tools / call_tool        │   (you!)     │
│  Desktop,   │      list_resources / read         │              │
│  Cursor,    │      list_prompts / get            │  - tools     │
│  smolagent) │                                     │  - resources │
└─────────────┘                                     │  - prompts   │
                                                    └──────────────┘
```

Three primitives:
- **Tools** — executable functions (have side effects). Like REST `POST`.
- **Resources** — read-only data the model can pull (files, db rows). Like REST `GET`.
- **Prompts** — reusable prompt templates the *user* invokes ("/summarize-arabic"). Like slash-commands.

---


## 1 · Setup

In [ ]:
%pip install -q --upgrade \
    "mcp>=1.2" \
    "fastmcp>=0.4" \
    "smolagents[mcp,litellm]>=1.13" \
    "langfuse>=2.55" \
    "httpx" "rich" "uvicorn" "pyngrok"

import os, getpass
if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

## 2 · Your first MCP server with **FastMCP**

`fastmcp` is to MCP what FastAPI is to HTTP — decorators, type hints, automatic schema. Run the cell below to write the server file, then **read it carefully** — every block teaches a primitive.

In [ ]:
# @title darija_server.py — a real MCP server
import base64
SERVER_B64 = "ZnJvbSBmYXN0bWNwIGltcG9ydCBGYXN0TUNQCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmltcG9ydCByZSwgdW5pY29kZWRhdGEsIGpzb24sIHVybGxpYi5yZXF1ZXN0CgptY3AgPSBGYXN0TUNQKCJkYXJpamEtdG9vbGtpdCIpCgojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgCBUT09MUyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCkBtY3AudG9vbApkZWYgdHJhbnNsaXRlcmF0ZSh0ZXh0OiBzdHIpIC0+IHN0cjoKICAgICIiIkNvbnZlcnQgQXJhYmljIHNjcmlwdCB0byBMYXRpbiAoQnVja3dhbHRlci1pc2gpLiBVc2VmdWwgZm9yIGxvZ3MuIiIiCiAgICB0YWJsZSA9IHN0ci5tYWtldHJhbnMoewogICAgICAgICLYpyI6ImEiLCLYqCI6ImIiLCLYqiI6InQiLCLYqyI6InRoIiwi2KwiOiJqIiwi2K0iOiI3Iiwi2K4iOiJraCIsItivIjoiZCIsCiAgICAgICAgItiwIjoiZGgiLCLYsSI6InIiLCLYsiI6InoiLCLYsyI6InMiLCLYtCI6InNoIiwi2LUiOiJTIiwi2LYiOiJEIiwi2LciOiJUIiwKICAgICAgICAi2LgiOiJaIiwi2LkiOiIzIiwi2LoiOiJnaCIsItmBIjoiZiIsItmCIjoicSIsItmDIjoiayIsItmEIjoibCIsItmFIjoibSIsCiAgICAgICAgItmGIjoibiIsItmHIjoiaCIsItmIIjoidyIsItmKIjoieSIsItmJIjoiYSIsItipIjoiYSIsItihIjoiYSIsCiAgICB9KQogICAgcmV0dXJuIHVuaWNvZGVkYXRhLm5vcm1hbGl6ZSgiTkZLQyIsIHRleHQpLnRyYW5zbGF0ZSh0YWJsZSkKCkBtY3AudG9vbApkZWYgZGV0ZWN0X2RpYWxlY3QodGV4dDogc3RyKSAtPiBkaWN0OgogICAgIiIiSGV1cmlzdGljIERhcmlqYSB2cyBNU0EgZGV0ZWN0b3IuIiIiCiAgICBkYXJpamFfbWFya2VycyA9IFsid2FjaCIsIndhc2giLCJiemFmIiwiYnplZiIsImdoYWRpIiwia2F5biIsIm1ha2F5biIsCiAgICAgICAgICAgICAgICAgICAgICAiZjdhbCIsImtpbWEiLCJkeWFsIiwibnRhIiwibnRpIiwia2hvdXlhIiwibWFzaGkiXQogICAgbXNhX21hcmtlcnMgICAgPSBbItil2YYiLCLZhNmD2YYiLCLYrdmK2KsiLCLYs9mI2YEiLCLZgtivIiwi2KfZhNiw2YoiLCLYp9mE2KrZiiIsItmE2KPZhiJdCiAgICB0ID0gdGV4dC5sb3dlcigpCiAgICBkID0gc3VtKDEgZm9yIG0gaW4gZGFyaWphX21hcmtlcnMgaWYgbSBpbiB0KQogICAgYSA9IHN1bSgxIGZvciBtIGluIG1zYV9tYXJrZXJzICAgIGlmIG0gaW4gdGV4dCkKICAgIGlmIGQgPT0gYSA9PSAwOgogICAgICAgIHJldHVybiB7ImRpYWxlY3QiOiJ1bmtub3duIiwiY29uZmlkZW5jZSI6MC4wLCJtYXJrZXJzIjpbXX0KICAgIGlmIGQgPiBhOgogICAgICAgIHJldHVybiB7ImRpYWxlY3QiOiJkYXJpamEiLCJjb25maWRlbmNlIjpkLyhkK2EpLAogICAgICAgICAgICAgICAgIm1hcmtlcnMiOlttIGZvciBtIGluIGRhcmlqYV9tYXJrZXJzIGlmIG0gaW4gdF19CiAgICByZXR1cm4geyJkaWFsZWN0IjoibXNhIiwiY29uZmlkZW5jZSI6YS8oZCthKSwKICAgICAgICAgICAgIm1hcmtlcnMiOlttIGZvciBtIGluIG1zYV9tYXJrZXJzIGlmIG0gaW4gdGV4dF19CgpAbWNwLnRvb2wKZGVmIGNsZWFuX2FyYWJpYyh0ZXh0OiBzdHIpIC0+IHN0cjoKICAgICIiIlN0cmlwIGRpYWNyaXRpY3MsIHRhdHdlZWwsIGFuZCByZWR1bmRhbnQgd2hpdGVzcGFjZS4iIiIKICAgIHRleHQgPSByZS5zdWIociJbXHUwNjRCLVx1MDY1Mlx1MDY3MFx1MDY0MF0iLCAiIiwgdGV4dCkKICAgIHJldHVybiByZS5zdWIociJccysiLCIgIiwgdGV4dCkuc3RyaXAoKQoKQG1jcC50b29sCmRlZiBoaWpyaV90b2RheSgpIC0+IHN0cjoKICAgICIiIlRvZGF5J3MgSGlqcmkgZGF0ZSB2aWEgQWxhZGhhbiBwdWJsaWMgQVBJLiIiIgogICAgdXJsID0gImh0dHBzOi8vYXBpLmFsYWRoYW4uY29tL3YxL2dUb0g/ZGF0ZT0iICsgZGF0ZXRpbWUubm93KCkuc3RyZnRpbWUoIiVkLSVtLSVZIikKICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwsIHRpbWVvdXQ9NSkgYXMgcjoKICAgICAgICBkYXRhID0ganNvbi5sb2FkcyhyLnJlYWQoKSkKICAgIGggPSBkYXRhWyJkYXRhIl1bImhpanJpIl0KICAgIHJldHVybiBmJ3toWyJkYXkiXX0ge2hbIm1vbnRoIl1bImVuIl19IHtoWyJ5ZWFyIl19IEFIJwoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgUkVTT1VSQ0VTIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKQG1jcC5yZXNvdXJjZSgiZGFyaWphOi8vc3RvcHdvcmRzIikKZGVmIHN0b3B3b3JkcygpIC0+IHN0cjoKICAgICIiIlRoZSAxMzM3QUkgRGFyaWphIHN0b3B3b3JkIGxpc3QuIiIiCiAgICBzdyA9IFsiYmFjaCIsImdoaXIiLCJnaGFkaSIsImtheW4iLCJrYXluYSIsIndhY2giLCJ3YXNoIiwiYnphZiIsImtpZmFjaCIsCiAgICAgICAgICAiZmluIiwiZmFjaCIsIjliZWwiLCJtbiIsImwiLCJmIiwiYiIsIjNsYSIsIm0zYSIsImJsYSIsImtvbCIsImtvbGxjaGkiXQogICAgcmV0dXJuICJcbiIuam9pbihzdykKCkBtY3AucmVzb3VyY2UoImRhcmlqYTovL2dyZWV0aW5ncyIpCmRlZiBncmVldGluZ3MoKSAtPiBzdHI6CiAgICAiIiJDb21tb24gRGFyaWphIGdyZWV0aW5ncywgdHJhbnNsaXRlcmF0ZWQgKyBBcmFiaWMuIiIiCiAgICByZXR1cm4ganNvbi5kdW1wcyhbCiAgICAgICAgeyJsYXRpbiI6InNhbGFtIiwiYXJhYmljIjoi2LPZhNin2YUiLCJtZWFuaW5nIjoiaGVsbG8ifSwKICAgICAgICB7ImxhdGluIjoibGFiYXMiLCJhcmFiaWMiOiLZhNin2KjYp9izIiwibWVhbmluZyI6ImhvdyBhcmUgeW91In0sCiAgICAgICAgeyJsYXRpbiI6ImJpa2hpciIsImFyYWJpYyI6Itio2K7ZitixIiwibWVhbmluZyI6ImZpbmUifSwKICAgICAgICB7ImxhdGluIjoic2hva3JhbiIsImFyYWJpYyI6Iti02YPYsdinIiwibWVhbmluZyI6InRoYW5rIHlvdSJ9LAogICAgXSwgZW5zdXJlX2FzY2lpPUZhbHNlKQoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAgUFJPTVBUUyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCkBtY3AucHJvbXB0CmRlZiBzdW1tYXJpemVfaW5fZGFyaWphKHRleHQ6IHN0cikgLT4gc3RyOgogICAgIiIiU3VtbWFyaXplIHRoZSBnaXZlbiB0ZXh0IGluIDMgc2VudGVuY2VzIG9mIERhcmlqYS4iIiIKICAgIHJldHVybiBmIkxraGFkIGhhZCBuYXNzIHcgM3RpbmkgcsOpc3Vtw6kgZiB0bGF0YSBqYW1hbHMgYiBkYXJpamE6XG5cbnt0ZXh0fSIKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtY3AucnVuKHRyYW5zcG9ydD0ic3RyZWFtYWJsZS1odHRwIiwgaG9zdD0iMC4wLjAuMCIsIHBvcnQ9ODc2NSkK"
with open("darija_server.py", "wb") as f:
    f.write(base64.b64decode(SERVER_B64))

print("✅ Wrote darija_server.py")
print("─" * 60)
print(open("darija_server.py").read())

In [ ]:
# @title Launch the server in the background
import subprocess, time, sys
proc = subprocess.Popen([sys.executable, "darija_server.py"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)
print("Server started, pid =", proc.pid)
print("Endpoint: http://127.0.0.1:8765/mcp")

## 3 · Connect from a smolagents client

`smolagents` >= 1.13 has a built-in MCP client. One line and *every* tool on the server becomes a regular smolagents tool.

In [ ]:
# @title MCP-aware agent
from smolagents import CodeAgent, InferenceClientModel
from smolagents.mcp_client import MCPClient

mcp_client = MCPClient({"url":"http://127.0.0.1:8765/mcp", "transport":"streamable-http"})
tools = mcp_client.get_tools()
print("Discovered tools:", [t.name for t in tools])

agent = CodeAgent(
    tools=list(tools),
    model=InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct"),
    additional_authorized_imports=["json"],
)

agent.run("3tini résumé sur had l\'jommla, w 9oli wach hia darija wlla 3arabia foS7a: "
          "'Hadchi li kayktbouh l\'youtoubeurs MA9roniche bzaf, mashi mhem.'")

**The agent just**:
1. Called `detect_dialect` → got `{dialect: "darija"}`
2. Called `clean_arabic` to normalize
3. Composed a final answer

Same agent code would work against **any** MCP server — yours, GitHub's, Stripe's, Notion's. That's the whole point.

---

## 4 · Connecting to **Claude Desktop** (or Cursor)

Edit your Claude Desktop `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "darija-toolkit": {
      "command": "python",
      "args": ["/full/path/to/darija_server.py"],
      "transport": "stdio"
    }
  }
}
```

Restart Claude Desktop → the 🔨 icon now shows your tools. Same JSON works for Cursor, Cline, and any MCP host. To switch from HTTP to stdio for local use, change the `mcp.run(...)` call at the bottom of the server file to `mcp.run(transport="stdio")`.

---


## 5 · Exposing your server publicly with `pyngrok` (for the leaderboard)

The mentors' evaluation agent runs in the cloud — it needs a public URL.

In [ ]:
# @title 🌐 Make your MCP server reachable for grading
from pyngrok import ngrok, conf
import os

if os.getenv("NGROK_TOKEN"):
    conf.get_default().auth_token = os.environ["NGROK_TOKEN"]

public = ngrok.connect(8765, "http")
print(f"\n🚀 PUBLIC MCP URL: {public.public_url}/mcp")
print("   Submit this URL on the leaderboard form.")

## 6 · 🏆 Challenge — *"Plug & Play"*

Build a **production MCP server** that:
1. Has at least **3 tools**, **1 resource**, **1 prompt**
2. Handles **errors gracefully** (return helpful error messages, don't crash)
3. Has **proper docstrings** (the LLM reads them)
4. Includes **at least one Darija-specific** capability
5. Works over **streamable-http** transport

### Auto-grader sketch

The mentors will run this against your URL:

```python
client = MCPClient({"url":"<YOUR_URL>","transport":"streamable-http"})
agent  = CodeAgent(tools=client.get_tools(), model=...)
score  = run_hidden_eval(agent)  # 30 mixed Darija/Arabic queries
```

### Bonus points
- 🎯 +20 if your server is **stateless** and reusable across calls
- 🎯 +20 if you ship a **GitHub repo + README** so other teams can fork it
- 🎯 +30 if you build a **Langfuse dashboard** showing your server's hot tools


## 7 · Sanity-check your server with the official inspector

```bash
npx @modelcontextprotocol/inspector python darija_server.py
```

Opens a web UI to manually call every tool, view schemas, debug transport issues. **Use this before submitting.**

---

## Recap

You learned:
- ✅ The 3 MCP primitives: tools, resources, prompts
- ✅ How to ship a server with `FastMCP` in <50 LoC
- ✅ How to consume MCP from `smolagents`, Claude Desktop, Cursor
- ✅ How to expose it publicly with ngrok
- ✅ Why MCP is the **universal connector** for the agent ecosystem

**Next →** `03_vision_detection.ipynb`: state-of-the-art vision with Florence-2, Qwen2.5-VL, and SAM 2.

> *"Write tools once, run them everywhere"* — the MCP promise.
